# AEF-BNG: Databricks Pipeline

This notebook runs the `aef-bng` distributed Spark pipeline on a Databricks cluster. It reprojects
AEF (Annual Feature Embedding) tiles from UTM to the British National Grid at 10m resolution and
writes the results to a Unity Catalog Delta table.

The Spark processing must be run on a Databricks cluster either through the Databricks environment
using Python (below), or via the example asset bundle. Once the processing has run, you can use
Databricks Connect to interact with your cluster via VS Code.

## How it works

1. **Tile index query**: The AEF STAC GeoParquet index on Source Cooperative S3 is queried with
   predicate pushdown. Only tiles overlapping the requested BNG bounds and years are returned.
2. **Grid enumeration**: The BNG extent is divided into 10km grid squares. Each square becomes a
   Spark task distributed across the cluster.
3. **Processing (per task)**: Windowed COG reads via `async-geotiff`, UTM→BNG reprojection with
   `rasterio`, merge at UTM zone boundaries (first-valid strategy), and extraction of one row per
   valid 10m cell.
4. **Arrow-native throughput**: `mapInArrow` processes chunks as PyArrow RecordBatches with a
   single `asyncio` event loop per partition.
5. **Delta output**: Results are written to a Unity Catalog Delta table with liquid clustering on
   `(year, bng_ref)` for spatial and temporal queries.

## Requirements

- **Runtime**: Databricks Runtime 17.3 LTS+ (Spark 4.0, Python 3.12)
- **Network**: Outbound access to `us-west-2.opendata.source.coop` (public S3, no credentials)
- **Permissions**: Unity Catalog `CREATE TABLE` / `INSERT` on the target schema
- **Cluster**: Standard_DS4_v2 with autoscaling (1–10 workers)

## Output schema

| Column | Type | Description |
|--------|------|-------------|
| `bng_ref` | string | 10-character BNG grid reference (10m cell) |
| `year` | smallint | Year of the AEF embedding |
| `A00`–`A63` | tinyint | 64 int8 embedding bands (dequantise to float with `aef_bng.dequantise.dequantise_spark`) |
| `geometry` | geometry | 10m × 10m polygon in EPSG:27700 |

## 1. Install the package

Install `aef-bng` on the cluster. Choose one of:

**From GitHub:**
```python
%pip install git+https://github.com/jordanp-dluhc/aef-bng.git
```

**From a pre-built wheel (if using the init script or cluster library):**
```bash
# Via uv (if cluster has the aef-bng init script):
uv pip install /path/to/aef_bng-*.whl
```

You can also install the Python wheel during cluster start up using an
[init script](https://docs.databricks.com/aws/en/init-scripts/). A template init script exists in
the repo for installing `uv` and an existing `aef-bng` Python wheel.

You can also run a job as a Databricks Asset Bundle, see the repo DAB template.

## 2. Configure the pipeline

Set the processing bounds, years, and output table. Adjust these for your area of interest.

- **Bounds**: EPSG:27700 (BNG) metres as `(minx, miny, maxx, maxy)`
- **Years**: Which annual AEF embeddings to process (available: 2017-2025)
- **Table**: Unity Catalog three-level name (`catalog.schema.table`)

Use backticks around catalog names containing special characters, e.g. `` `catalog-with-character`.schema.table ``.

In [1]:
CATALOG = "catalog"
SCHEMA = "data"
TABLE = "aef_embeddings"

YEARS = [2025]
BOUNDS = (508848, 163362, 553002, 196746)  # London, UK

TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"


print(f"Target table:      {TABLE_NAME}")
print(f"Years:             {YEARS}")
print(f"Bounds:            {BOUNDS}")

Target table:      catalog.data.aef_embeddings
Years:             [2025]
Bounds:            (508848, 163362, 553002, 196746)


In [4]:
from aef_bng.config import AEFBNGConfig

config = AEFBNGConfig(
    years=YEARS,
    bounds=BOUNDS,
    table_name=TABLE_NAME,
)
config

AEFBNGConfig(years=[2025], bounds=(508848, 163362, 553002, 196746), chunk_size=10000, output_path='./aef_bng_output', resampling='nearest', max_workers=4, table_name='catalog.data.aef_embeddings')

_Note_: the `output_path` and `max_workers` are ignored when processing with Spark.

## 3. Preview the processing grid

Check how many 10km chunks will be processed. Each chunk produces up to 1,000,000 rows
(1000 × 1000 pixels at 10m).

In [7]:
from aef_bng.grid import BNGOutputGrid

grid = BNGOutputGrid(config.bounds, config.chunk_size)
chunks = grid.enumerate_chunks()

num_tasks = len(chunks) * len(config.years)
print(f"10km chunks:      {len(chunks)}")
print(f"Years:            {len(config.years)}")
print(f"Max possible rows: {num_tasks * 1_000_000:,}")
print()
print("Sample chunks:")
for c in chunks[:10]:
    print(f"  {c.bng_10km_ref}  bounds={c.bounds_bng}")

10km chunks:      24
Years:            1
Max possible rows: 24,000,000

Sample chunks:
  TQ06  bounds=(500000, 160000, 510000, 170000)
  TQ16  bounds=(510000, 160000, 520000, 170000)
  TQ26  bounds=(520000, 160000, 530000, 170000)
  TQ36  bounds=(530000, 160000, 540000, 170000)
  TQ46  bounds=(540000, 160000, 550000, 170000)
  TQ56  bounds=(550000, 160000, 560000, 170000)
  TQ07  bounds=(500000, 170000, 510000, 180000)
  TQ17  bounds=(510000, 170000, 520000, 180000)
  TQ27  bounds=(520000, 170000, 530000, 180000)
  TQ37  bounds=(530000, 170000, 540000, 180000)


## 4. Preview the tile index

Query the AEF STAC GeoParquet index to see which COG tiles overlap the
requested bounds. This is the same query the pipeline runs internally — it
goes directly to Source Cooperative S3 with predicate pushdown.

In [8]:
from aef_bng.index import AEFBNGIndex

index = AEFBNGIndex()
tiles_gdf = index.load_for_bounds(config.bounds, config.years)

print(f"Tiles matching bounds + years: {len(tiles_gdf)}")
print(f"Columns: {list(tiles_gdf.columns)}")
tiles_gdf.head()

Tiles matching bounds + years: 2
Columns: ['type', 'stac_version', 'id', 'proj:epsg', 'datetime', 'links', 'assets', 'collection', 'bbox', 'geometry']


,type,stac_version,id,proj:epsg,datetime,links,assets,collection,bbox,geometry
0,Feature,1.1.0,75939,EPSG:32631,2025-01-01 00:00:00+00:00,[],{'data': {'href': 's3://us-west-2.opendata.sou...,None,"{'xmin': 0.0, 'ymin': 50.98534191223527, 'xmax...","POLYGON ((0.66492 50.98534, 0.66492 51.73653, ..."
1,Feature,1.1.0,71463,EPSG:32630,2025-01-01 00:00:00+00:00,[],{'data': {'href': 's3://us-west-2.opendata.sou...,None,"{'xmin': -0.6649247831313557, 'ymin': 50.98534...","POLYGON ((0 50.98534, 0 51.73653, -0.66492 51...."


## 5. Run the pipeline

This distributes chunk processing across the cluster using `mapInArrow`.

**Driver side:**
- Tile index queried from S3, pickled, and broadcast to executors
- DataFrame of `(chunk, year)` combinations created and repartitioned (~3 chunks per partition)

**Executor side (per partition):**
- STAC GeoParquet Index deserialised once
- Single `asyncio` event loop for all S3 reads in the partition
- For each chunk: windowed `async-geotiff` COG read → reproject to BNG → merge overlapping tiles
    → extract pixels
- Results yielded as Arrow `RecordBatches`

**Output:**
- Written to Delta with `optimizeWrite` and `autoCompact` enabled
- Liquid clustering applied on `(year, bng_ref)`

Monitor progress in the Spark UI, look for the `mapInArrow` stage.

_Note_: ensure your running the following command on Databricks directly (i.e. no via Databricks Connect).

In [ ]:
from aef_bng.spark import process_with_spark

process_with_spark(config)

Once it's run, cluster by the year and BNG reference.

_Note_: the example below uses [Databricks Connect](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/) from VS Code.

In [ ]:
from aef_bng.utils import get_or_create_spark_session

spark = get_or_create_spark_session()

spark.sql("ALTER TABLE `catalog`.schema.table CLUSTER BY (year, bng_ref)")                                                                                
                                                                                                                                                            
spark.sql("OPTIMIZE `catalog`.schema.table")  

## 6. Verify the output

Check row counts, schema, and clustering status.

In [ ]:
df = spark.table(TABLE_NAME)

print("Schema:")
df.printSchema()
print(f"Total rows: {df.count():,}")

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

df.select(
    F.count("*").alias("total_rows"),
    F.min(F.length("bng_ref")).alias("min_ref_len"),
    F.max(F.length("bng_ref")).alias("max_ref_len"),
    F.countDistinct("year").alias("distinct_years"),
).show()

In [ ]:
df.groupBy("year").count().orderBy("year").show()

In [ ]:
# Confirm liquid clustering is active
detail = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}")
detail.select("clusteringColumns").show(truncate=False)

## 7. Spatial queries

The `geometry` column supports Databricks spatial functions. Liquid clustering
on `(year, bng_ref)` means queries filtering on these columns skip irrelevant
files automatically.

### Query by bounding box

In [ ]:
from pyspark.sql import functions as F

df = spark.table(TABLE_NAME)

result = df.filter(
    F.expr(
        "ST_Intersects(geometry, ST_GeomFromWKT("
        "'POLYGON ((532322 176872, 532322 181313, 526975 181313, 526975 176872, 532322 176872))',"
        " 27700))"
    )
)

print(f"Cells in bbox: {result.count():,}")
result.select("bng_ref", "year", "A00", "A01", "A02").show(5, truncate=False)

## 8. Visualise

There's some helper utility functions for converting to and from Spark to GeoPandas.

We'll use the quick RGB as the [local notebook](https://github.com/jordanp-dluhc/aef-bng/blob/main/notebooks/aef_bng_local_example.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from aef_bng.dequantise import dequantise_spark
from aef_bng.utils import spark_to_geopandas


def plot_embedding_rgb(
    gdf,
    r_band="A00",
    g_band="A01",
    b_band="A02",
    title=None,
    figsize=(12, 10),
):
    """Plot an RGB composite from three AEF embedding bands.

    Renders the actual 10m BNG polygon geometry, coloured by normalising
    the selected bands to [0, 1] and mapping to RGB.
    """
    # Extract band values
    r = gdf[r_band].to_numpy().astype(np.float64)
    g = gdf[g_band].to_numpy().astype(np.float64)
    b = gdf[b_band].to_numpy().astype(np.float64)

    # Normalise each channel to [0, 1] with 2-98 percentile stretch
    def norm(arr):
        lo, hi = np.nanpercentile(arr, [2, 98])
        return np.clip((arr - lo) / (hi - lo + 1e-10), 0, 1)

    rgb = np.stack([norm(r), norm(g), norm(b)], axis=-1)

    # Build a per-polygon colour list and plot with geopandas
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    gdf.plot(ax=ax, color=[tuple(c) for c in rgb], linewidth=0, antialiased=True)
    ax.set_axis_off()
    ax.set_aspect("equal")
    ax.set_title(title or f"AEF Embedding RGB: R={r_band}, G={g_band}, B={b_band}", fontsize=18)
    plt.tight_layout()
    return fig, ax


result = dequantise_spark(result)
gdf = spark_to_geopandas(result)

year = gdf.iloc[0].year

fig, ax = plot_embedding_rgb(
    gdf,
    r_band="A0",
    g_band="A01",
    b_band="A02",
    figsize=(14, 14),
)
plt.show()

## 9. Scaling up

### Wales
```python
config = AEFBNGConfig(
    years=[2025],
    bounds=(153325,157362,381182,400529),
    table_name="my_catalog.my_schema.aef_wales",
)
```

### All of Great Britain
```python
from aef_bng.constants import BNG_BOUNDS

config = AEFBNGConfig(
    years=[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    bounds=BNG_BOUNDS,  # (0, 0, 700_000, 1_300_000)
    table_name="my_catalog.my_schema.aef_bng_gb",
)
```

---

## Run via DAB

For scheduled/CI runs, use the Databricks Asset Bundle:

```bash
# Build wheel and deploy
make build

# Trigger with custom parameters
databricks bundle run aef_bng_pipeline -t dev \
    --params bounds=508848,163362,553002,196746 \
    --params years=2024,2025 \
    --params "table_name=\`catalog\`.schema.table"
```

See `databricks.template.yml` in the repo root for the example bundle configuration.